# Adjoint capture analysis: streamflow and lake capture

When a new well starts pumping, the water it produces has to come from
somewhere. Some of it is released from aquifer storage, and the rest is
**capture** — water that would otherwise have discharged to a stream or a lake,
or that the stream and lake now lose to the aquifer. This notebook computes
streamflow capture and lake capture for the synthetic-valley model using
**adjoint-state sensitivity analysis** with
[mf6adj](https://github.com/INTERA-Inc/mf6adj).

By the end of this notebook you will be able to:

- write an mf6adj **performance measure** for the flow through a boundary package,
- run a forward and an adjoint solve and read the resulting sensitivities,
- turn the sensitivity of a stream or lake flux to a well rate into a **capture
  fraction**,
- map where in the aquifer that capture is most sensitive to hydraulic
  conductivity, and
- check the adjoint result with two model runs.

## What an adjoint sensitivity is

The direct way to find out how sensitive a model result is to a parameter is to
change the parameter and run the model again. That costs one model run per
parameter, so a sensitivity map over a 5-layer, 40-by-25 grid would cost
thousands of runs.

The adjoint approach turns the problem around. You first name the single model
output you care about — the **performance measure** — and then solve the flow
equations backward in time once. That one backward solve returns the
sensitivity of that measure to every parameter in every cell: hydraulic
conductivity, storage, recharge, boundary conductance, and well rates.

The trade-off is the mirror image of the direct approach: one backward solve per
measure, rather than one forward run per parameter. mf6adj drives MODFLOW 6
through its Application Programming Interface (**API**) to collect the matrices
it needs, so the model itself is unmodified.

The method and its verification are described in Hayek and others (2025),
*MF6-ADJ: A Non-Intrusive Adjoint Sensitivity Capability for MODFLOW 6*,
Groundwater 63(6), 874–888.

Import the packages this notebook uses. `mf6adj` provides the adjoint solver and
`mf6_adj_helpers` collects the workspace setup shared by the two adjoint
notebooks.

In [ ]:
import flopy
import matplotlib as mpl
import matplotlib.pyplot as plt
import mf6_adj_helpers as adjh
import mf6adj
import numpy as np
from mf6_notebook_helpers import find_mf6_libraries, length_name, to_display, unit

Locate the MODFLOW 6 shared library and executable in the active environment.
mf6adj drives the model through the shared library (`libmf6`), so both paths are
needed: the executable for the ordinary forward runs and the library for the
adjoint.

In [ ]:
lib_name, mf6_exe = find_mf6_libraries()
print(f"library:    {lib_name.name}")
print(f"executable: {mf6_exe.name}")

## Prepare the model

Use the **advanced** synthetic-valley model at an annual sampling frequency. It
is in meters and days and represents the valley with the advanced hydrologic
packages: streamflow routing (**SFR**) for the river, a lake (**LAK**) with no
outlet, unsaturated-zone flow (**UZF**) for infiltration and evapotranspiration
above the water table, seepage drains (**DRN**) that remove groundwater where
the water table reaches the land surface, and the water mover (**MVR**), which
carries UZF runoff and drain water to the lake or the stream. The 21 stress
periods start with a long steady spin-up followed by 20 annual periods.

The model needs no preparation for the adjoint. The production wells are
multi-aquifer wells (**MAW**), whose heads MODFLOW 6 solves along with the
aquifer heads, so each well is already one of the rows of the matrix mf6adj
differentiates and nothing has to be added to the system for it. SFR and LAK are
different: MODFLOW 6 solves them separately from the aquifer heads and passes the
two back and forth until they agree, so mf6adj borders the adjoint system with
their own equations. UZF and the mover are not differentiated at all, which is
where the warnings further down come from.

In [ ]:
ws = adjh.prepare_model("adj-capture", variant="advanced", prediction_rate=0.0)
adjh.run_model(ws, mf6_exe)

sim = flopy.mf6.MFSimulation.load(sim_ws=str(ws), verbosity_level=0)
gwf = sim.get_model()
nper = sim.tdis.nper.data
print(f"stress periods: {nper}")
print(f"packages:       {', '.join(sorted(gwf.package_names))}")

This first run is the **baseline**: the prediction well is present but pumps at a
rate of zero. The sensitivities describe how the model responds to a small change
away from this baseline, which is what makes the capture fraction a property of
the aquifer and the well's position rather than of one particular pumping rate.

### Look at the model

Map the features that matter for capture: the lake cells, the stream reaches,
the seepage drains, the two production wells, and the prediction well. The
prediction well's cell is read from the prediction well package rather than
typed in, and its distance to the stream and the lake is printed.

In [ ]:
pred_cell = adjh.package_cells(gwf, "prediction")[0]  # zero-based (layer, row, column)
prod_cells = adjh.package_cells(gwf, "pwell")
sfr_cells = adjh.package_cells(gwf, "sfr-1")
lak_cells = adjh.package_cells(gwf, "lak-1")
drn_cells = adjh.package_cells(gwf, "drn-1")
nrow, ncol = gwf.modelgrid.nrow, gwf.modelgrid.ncol
print(f"prediction well cell: {pred_cell}")

xc, yc = gwf.modelgrid.xcellcenters, gwf.modelgrid.ycellcenters
px, py = xc[pred_cell[1], pred_cell[2]], yc[pred_cell[1], pred_cell[2]]
print("distance from the prediction well to the nearest")
for label, cells in (("stream reach", sfr_cells), ("lake cell", lak_cells)):
    d = min(np.hypot(xc[i, j] - px, yc[i, j] - py) for _, i, j in cells)
    print(f"  {label:13s} {to_display(d):7,.0f} {unit(plain=True)}")

# 1 seepage drain, 2 lake, 3 stream; the lake and stream cells are not drained
features = np.zeros((nrow, ncol))
for cells, value in ((drn_cells, 1), (lak_cells, 2), (sfr_cells, 3)):
    for _, i, j in cells:
        features[i, j] = value
feature_colors = ["wheat", "tab:blue", "tab:cyan"]

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
mm.plot_array(
    np.ma.masked_equal(features, 0),
    cmap=mpl.colors.ListedColormap(feature_colors),
    vmin=0.5,
    vmax=3.5,
)
mm.plot_grid(lw=0.2, color="0.6")
for k, i, j in prod_cells:
    ax.plot(xc[i, j], yc[i, j], "ko", ms=7)
ax.plot(px, py, "r*", ms=16, label="prediction well")
ax.plot([], [], "ko", ms=7, label="production wells")
for color, label in zip(feature_colors, ("seepage drains", "lake", "stream")):
    ax.plot([], [], "s", color=color, mec="0.4", label=label)
ax.legend(loc="upper right", fontsize=8)
# label the axes in display units; the model grid stays in meters
ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
ax.set_xlabel(f"x, in {length_name()}")
ax.set_ylabel(f"y, in {length_name()}")
ax.set_title("Synthetic valley: surface-water features and wells")

**What to look for.** The lake fills the northwest of the valley up to its
northern edge, and the stream runs south from below it to the southern edge.
Seepage drains cover the rest of layer 1. The prediction well (red star) is in
the south, in layer 5 at the bottom of the lower aquifer, 1,067 m (3,500 ft) from
the nearest stream reach and 3,002 m (9,849 ft) from the nearest lake cell.
Expect the stream to supply most of the captured water and the lake very little.

## Define the performance measures

An mf6adj performance measure lists the model outputs to combine. mf6adj reads
it from a plain-text block with one line per entry:

```
# kper kstp layer row column  package  form  weight  observed
  21    1     1    23    9    sfr-1   direct  1.0    -1.0e+30
```

The `package` field is either `head` — for the head in that cell — or the name
of a boundary package from the model name file, which selects the flow between
that package and the cell. `direct` means the measure is the value itself rather
than a residual against an observation, so the `observed` field is unused.

You do not have to format those lines yourself. Give
`mf6adj.write_performance_measures()` each measure as a dictionary of columns:
the cells in `cellid`, exactly as FloPy returns them, and the time in `kper` and
`kstp`. All of these are **zero-based**, as they are everywhere in FloPy, and
the writer converts them to the **one-based** indices of the file. `pm_form`,
`weight`, and `obsval` are spelled out below so you can see every column, but
the values are the defaults, and the other adjoint notebooks leave them out.
The file is written as ascii text unless you pass `format="hdf5"`.

Build two measures at the last stress period: `swgw` sums the exchange between
the stream and the aquifer over all 18 stream reaches, and `lakegw` sums the
exchange between the lake and the aquifer over all 153 lake connections. The
lake sits on top of the aquifer, so every connection is through the lake bed: 142
to layer 1 and 11 to layer 2, where layer 1 has no thickness beneath the deepest
part of the lake. Together the two measures are the surface-water flow that
pumping can capture. The seepage drains are an outlet too; the water budget at
the end of the notebook shows they carry no water in the final period, so they
need no measure here.

In [ ]:
last = nper - 1  # zero-based
print(f"stream reaches:   {len(sfr_cells)}")
print(f"lake connections: {len(lak_cells)}")

measures = {}
for name, package, cells in (
    ("swgw", "sfr-1", sfr_cells),
    ("lakegw", "lak-1", lak_cells),
):
    measures[name] = {
        "cellid": cells,
        "kper": last,
        "kstp": 0,  # every period is a single time step
        "pm_type": package,
        "pm_form": "direct",
        "weight": 1.0,
        "obsval": -1.0e30,
    }
adj_file = mf6adj.write_performance_measures(ws / "capture.adj", measures)
print(adj_file.read_text()[:260] + "...")

## Solve the forward and adjoint problems

Create the `mf6adj.Mf6Adj` object with the measure file and the shared library.
`solve_forward_model()` runs MODFLOW 6 through the API and saves the solution
matrix and boundary terms at every time step; `solve_adjoint()` then sweeps
backward through those saved time steps, once per measure. Always call
`finalize()` to release the library. Set `logging_level="WARNING"` to keep the
per-time-step progress messages out of the notebook.

`WARNING` is the level to keep, though, because mf6adj reports there when part
of a package's derivative is not formed. This model draws warnings about UZF,
the mover, and the lake.

In [ ]:
adj = mf6adj.Mf6Adj(
    adj_file.name,
    str(lib_name),
    logging_level="WARNING",
    working_directory=str(ws),
)
adj.solve_forward_model()
sensitivities = adj.solve_adjoint()
adj.finalize()

print(f"measures solved: {', '.join(sensitivities)}")
print(f"parameters:      {', '.join(sensitivities['swgw'].columns)}")

**What to look for.** The first warning names the two packages mf6adj forms no
terms for at all, UZF and the water mover. Their exchange with the aquifer is
carried by the flow matrix, so the head sensitivities account for it, but
neither package's response to pumping enters a derivative. UZF calculates how
much infiltration reaches the water table and how much is lost to
evapotranspiration on the way, and the two-run check at the end of the notebook
shows that pumping changes that split. The second warning, repeated for each
measure, names the inflow the mover routes into the lake, which follows the
state of the package supplying it. The stream draws no warning of its own.

Each measure returns a table with one row per model cell and one column per
parameter — hydraulic conductivity (`k11`, `k33`), the two storage properties
(specific storage `ss` and specific yield `sy`), recharge, the elevation and
conductance of the seepage drains, the stream and lake stage and conductance,
and the well rate (`wel6_q`). Every one of those columns came from the one
backward solve.

## Read the capture fractions

The sensitivity of a stream or lake flux to a well rate is the capture
fraction itself: it is the change in surface-water flow per unit of pumping, so a value
of -1 means every unit pumped is taken from that feature and 0 means none of it
is.

Read the `wel6_q` sensitivity at the prediction well cell. Sum the per-period
sensitivities over the periods the well actually pumps (12 through 21), because
each period's value is that period's contribution to the measure at the final
period.

In [ ]:
active = range(11, nper)  # zero-based periods the prediction well pumps
capture = {}
for measure in ("swgw", "lakegw"):
    capture[measure] = adjh.total_sensitivity(
        ws, measure, "wel6_q", cell=pred_cell, periods=active
    )

print(f"streamflow capture fraction: {capture['swgw']:8.4f}")
print(f"lake capture fraction:       {capture['lakegw']:8.4f}")
print(f"total surface-water capture: {sum(capture.values()):8.4f}")

**What to look for.** The stream's capture fraction is -1.032. It is negative
because the well rate is itself negative: a higher pumping rate makes the stream
give up more water. Its size says the stream gives up slightly more water than
the well pumps. The lake's is 0.013, the other sign: with the well pumping, the
lake gives up less water to the aquifer than before, by 0.013 of the pumping.
The two together are -1.019. A stream share above 1 means pumping also takes
water from somewhere else that the stream ends up replacing, which the two-run
check at the end of the notebook traces.

### Map capture for a well anywhere in the aquifer

The `wel6_q` sensitivity is not only defined at the prediction well. mf6adj
returns it in every cell, so the same backward solve says what fraction of a
well's pumping would be captured from the stream or the lake wherever the well
were placed. Map the negative of the composite `wel6_q` sensitivity for each
measure, which is the capture fraction for a well pumping over every stress
period, in layer 1, the water table, and layer 5, the layer the prediction well
is in. Layer 3 is left out: outside the clay it has no thickness, and water
passes straight through it.

Each measure has its own color scale, shared by its two layers. A negative lake
capture is a lake giving up less water to the aquifer under pumping than it did
before. The gray cells in layer 1 are where layer 1 has no thickness beneath the
deepest part of the lake, which connects to layer 2 there instead.

In [ ]:
layers = (0, pred_cell[0])  # zero-based: the water table and the well's layer
capture_maps = {
    measure: -adjh.composite_sensitivity(ws, measure, "wel6_q")
    for measure in ("swgw", "lakegw")
}
titles = {"swgw": "Streamflow capture", "lakegw": "Lake capture"}
idomain = gwf.dis.idomain.array

fig, axd = plt.subplot_mosaic(
    [[f"{measure}{k}" for k in layers] for measure in capture_maps],
    figsize=(9, 13),
    constrained_layout=True,
)
for measure, capture_map in capture_maps.items():
    # each measure on its own scale, shared by its two layers
    values = np.concatenate([capture_map[k][idomain[k] > 0] for k in layers])
    for k in layers:
        ax = axd[f"{measure}{k}"]
        mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=k)
        cb = mm.plot_array(
            capture_map[k], cmap="viridis", vmin=values.min(), vmax=values.max()
        )
        # cells with no thickness, where the lake sits on layer 2, in gray
        mm.plot_ibound(color_vpt="0.6")

        # the stream and lake are in layer 1, so outline their cells rather than
        # plotting them with plot_bc
        for cells, color in ((sfr_cells, "tab:cyan"), (lak_cells, "white")):
            rows = [i for _, i, _ in cells]
            cols = [j for _, _, j in cells]
            ax.plot(xc[rows, cols], yc[rows, cols], "s", mfc="none", mec=color, ms=4)
        ax.plot(px, py, "r*", ms=14)
        ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
        ax.set_xlabel(f"x, in {length_name()}")
        ax.set_ylabel(f"y, in {length_name()}")
        ax.set_title(f"{titles[measure]}, layer {k + 1}")
    fig.colorbar(
        cb,
        ax=[axd[f"{measure}{k}"] for k in layers],
        shrink=0.8,
        label="capture fraction (-)",
    )

for measure, capture_map in capture_maps.items():
    for k in layers:
        values = capture_map[k][idomain[k] > 0]
        print(
            f"{titles[measure]:18s} layer {k + 1}: "
            f"{values.min():7.3f} to {values.max():7.3f}"
        )
    difference = np.abs(capture_map[layers[0]] - capture_map[layers[1]])
    both = (idomain[layers[0]] > 0) & (idomain[layers[1]] > 0)
    print(
        f"{'':18s} layers {layers[0] + 1} and {layers[1] + 1} differ by at most "
        f"{difference[both].max():.3f}"
    )

**What to look for.** Stream capture is above 1 everywhere, from 1.024 to 1.208
in layer 1 and 1.026 to 1.157 in layer 5. It is lowest in the south, around the
lower stream and the prediction well, and highest beneath the lake. Lake capture
is negative everywhere, from -0.260 to -0.002 in layer 1 and -0.168 to -0.002 in
layer 5, and most negative beneath the lake. The two maps are close to mirror
images: a well near the lake reduces the lake's leakage, and the stream makes up
the difference. Layers 1 and 5 differ by at most 0.059 for the stream and 0.098
for the lake.

A well near the lake takes less water from it, not more, because the lake has no
outlet. Pumping in a given year draws water from the lake, but nothing replaces
it: the stage falls, the lake shrinks, and it leaks less to the aquifer in every
later year. The composite sensitivity adds up all the years of pumping, and the
years of lowered stage outweigh the one year of extra leakage.

The streamflow-augmentation example in
[mf6-api-e](mf6-api-e-streamflow-augmentation.ipynb) pumps this well into the
stream. At a capture fraction of 1.032, ten years of pumping draw about as much
water out of the stream through its bed in the final year as the well pumps, so
little of the water pumped from this well into the stream stays in it.

### How capture builds up through time

The per-period sensitivities of a measure defined at the final period show how
much pumping in each earlier period contributes to the measure. Plot the stream
capture contributed by each stress period.

In [ ]:
per_period = adjh.period_sensitivity(ws, "swgw", "wel6_q")
contrib = np.array([per_period[kper][pred_cell] for kper in sorted(per_period)])
print("share of the pumping-period capture from")
for kper in (last, last - 1, last - 2):
    print(f"  period {kper + 1}: {contrib[kper] / contrib[list(active)].sum():.3f}")
print(f"  periods before the well starts: {contrib[:11].sum() / contrib.sum():.1e}")

fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.bar(np.arange(1, nper + 1), -contrib, color="tab:cyan", edgecolor="k", lw=0.4)
ax.set_xlabel("stress period")
ax.set_ylabel("contribution to streamflow capture (-)")
ax.set_xticks(range(2, nper + 1, 2))
ax.set_title("Where the final-period capture comes from in time")

**What to look for.** The last stress period supplies 0.773 of the capture:
pumping in the year the measure is evaluated accounts for most of it. The year
before it supplies 0.132 and the one before that 0.040, and each earlier year
less again, which is the aquifer's memory of past pumping. The periods before the
well turned on would add 0.008 for a well that had pumped through them.

## Map where the stream exchange is most sensitive

The same backward solve produced the sensitivity of the stream measure to
hydraulic conductivity in every cell. Map the composite `k11` sensitivity for the
top layer to see which parts of the aquifer control how much water the stream
gains from the aquifer in the final year. The sensitivity is a change in flow per
unit of conductivity.

In [ ]:
# the change in stream exchange per unit of conductivity is a flow per rate,
# which is an area
k11_sens = to_display(adjh.composite_sensitivity(ws, "swgw", "k11"), "area")
top = k11_sens[0][idomain[0] > 0]
ranked = np.sort(np.abs(top))[::-1]
share = np.cumsum(ranked) / ranked.sum()
print(
    f"largest 10% of cells carry {share[int(0.1 * ranked.size)]:.2f} of the total, "
    f"largest 25% carry {share[int(0.25 * ranked.size)]:.2f}"
)

fig, ax = plt.subplots(figsize=(6, 8), constrained_layout=True)
mm = flopy.plot.PlotMapView(model=gwf, ax=ax, layer=0)
vmax = np.percentile(np.abs(top), 99)
cb = mm.plot_array(k11_sens[0], cmap="RdBu_r", vmin=-vmax, vmax=vmax)
mm.plot_bc("SFR", color="tab:cyan")
mm.plot_ibound(color_vpt="0.6")
ax.plot(px, py, "k*", ms=14)
ax.xaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
ax.yaxis.set_major_formatter(lambda v, _: f"{to_display(v):,.0f}")
ax.set_xlabel(f"x, in {length_name()}")
ax.set_ylabel(f"y, in {length_name()}")
plt.colorbar(
    cb,
    ax=ax,
    shrink=0.5,
    label=f"change in flow per {unit('rate')}, in {unit('flow')}",
)
ax.set_title("Sensitivity of the stream exchange to layer-1 conductivity")

**What to look for.** The sensitivity is concentrated in a band between the lake
and the head of the stream. The measure is the flow from the stream into the
aquifer, which is negative where the stream gains, so a red cell is one where
higher conductivity would reduce the stream's gain and a blue cell, east of the
lake, one where it would increase it. A tenth of the cells carry 0.48 of the
total sensitivity and a quarter of them carry 0.74, so the adjoint shows where
the next field measurement would be most useful. This is the sensitivity of the
stream exchange itself, not of the prediction well's capture fraction, which is
why the cells around the well carry little.

## Check the adjoint with two model runs

The direct way to get a capture fraction is to run the model twice, once with
the well off and once with it on, and difference the budgets. That gives one
number per run, but it is an independent check on the adjoint.

Run the model again with the prediction well pumping at a small rate. Keep the
rate small: the adjoint returns a derivative evaluated at zero pumping, so the
difference only matches it in the limit of a small perturbation. Tighten the
solver's head-change criterion at the same time, because the differences in lake
flow are small enough to be lost within the solver's convergence tolerance.
Difference every budget term rather than only the two measures, so the whole of
the pumping is accounted for, and list the periods the seepage drains flow in.
Save the unsaturated-zone budget as well (`uzf_budget=True`), to see what
pumping changes above the water table, and read the final lake stage from the
lake observations.

In [ ]:
dq = -100.0  # m^3/d, small enough to stay in the linear range

ws_base = adjh.prepare_model(
    "adj-capture-q0", prediction_rate=0.0, outer_dvclose=1.0e-10, uzf_budget=True
)
ws_pert = adjh.prepare_model(
    "adj-capture-dq", prediction_rate=dq, outer_dvclose=1.0e-10, uzf_budget=True
)
for w in (ws_base, ws_pert):
    adjh.run_model(w, mf6_exe)

print("measure    adjoint   two-run   difference")
for measure, term in (("swgw", "SFR(SFR-1)"), ("lakegw", "LAK(LAK-1)")):
    diff = (
        adjh.budget_net(ws_pert, term)[last] - adjh.budget_net(ws_base, term)[last]
    ) / dq
    print(
        f"{measure:8s} {capture[measure]:9.4f} {diff:9.4f} "
        f"{100 * abs(diff - capture[measure]) / abs(capture[measure]):9.1f}%"
    )

print("\nshare of the pumping each budget term supplies in the final period")
sources = adjh.pumping_sources(ws_base, ws_pert, dq, last)
for term, share in sources.items():
    print(f"  {term:20s} {share:8.4f}")
print(f"  {'total':20s} {sum(sources.values()):8.4f}")

# seepage reaches the drains only when the water table is near the land surface
drained = -(
    adjh.budget_net(ws_base, "DRN(DRN-1)")
    + adjh.budget_net(ws_base, "DRN-TO-MVR(DRN-1)")
)
wet = {kper + 1: q for kper, q in enumerate(drained) if q > 0.01}
print("\nseepage-drain outflow without the well, by stress period")
for kper, q in wet.items():
    print(f"  {kper:2d} {to_display(q, 'flow'):10,.0f} {unit('flow', plain=True)}")
if not wet:
    print("  none in any period")

# what pumping changes in the unsaturated zone and at the lake
print("\nchange in the unsaturated-zone budget per unit pumped, final period")
uzf_base = adjh.csv_row(ws_base, "sv.uzf.csv", last)
uzf_pert = adjh.csv_row(ws_pert, "sv.uzf.csv", last)
for column in uzf_base.index:
    if column in ("time", "PERCENT_DIFFERENCE") or column.startswith("TOTAL"):
        continue
    if uzf_base[column] != 0.0 or uzf_pert[column] != 0.0:
        print(f"  {column:18s} {(uzf_pert[column] - uzf_base[column]) / -dq:8.4f}")
stage = [
    to_display(adjh.csv_row(w, "sv.lake.obs.csv", last)["LAKE-STAGE"])
    for w in (ws_base, ws_pert)
]
print(
    f"\nfinal lake stage: {stage[0]:.3f} {unit(plain=True)} without the well, "
    f"{stage[1]:.3f} {unit(plain=True)} with it"
)

**What to look for.** The adjoint and the two-run difference agree to 5.9
percent for the stream (-1.032 compared to -1.093). Repeating the pair of runs
at a tenth and at ten times this rate changes the differences by 0.001 or less,
so the gap is not a linearization error.

The budget says what the gap is. Every unit the well pumps in the final year is
supplied by the stream (1.0927) and aquifer storage (0.0139: 0.0079 from the
aquifer's elastic storage and 0.0060 from draining at the water table), less the
0.0064 the lake no longer gives up and the 0.1002 less recharge UZF delivers to the water
table, and those add to 1.0000. The unsaturated-zone budget shows where that
recharge goes. Infiltration does not change, evapotranspiration from the
unsaturated zone rises by 0.1027 of the pumping, and recharge to the water table
falls by 0.1002. This model takes evapotranspiration only from the unsaturated
zone, within 3.048 m (10 ft) of the land surface, so a lower water table leaves
more unsaturated ground within reach of the roots and less of the infiltration
gets through. The stream makes up that loss as well as the pumping. mf6adj
reported at the start of the adjoint solve that it forms no terms for UZF, so the
adjoint leaves out the recharge term, which accounts for part of the stream's
gap, 0.061.

The lake differs by 50.7 percent, 0.0130 compared to 0.0064, but both have the
same sign and are under 2 percent of the stream's share, so this well's water
comes from the stream whichever is used. The lake's final stage is 4.790 m
(15.715 ft) without the well and 4.787 m (15.705 ft) with it: the small drop
that makes the lake leak less.

The seepage drains carry nothing in the final year, with or without the well, so
they supply none of its water. Without the well they flow only in periods 1, 2,
14, and 16, when the water table reaches them, and at most 1,391 m³/d
(49,137 ft³/d).

An adjoint sensitivity answers the question its formulation encodes, and mf6adj
reports which parts of that formulation are approximate. Where the difference
changes the answer, the two-run difference is the number to trust.

## Recap

- A **performance measure** names the model output you care about; one backward
  adjoint solve returns its sensitivity to every parameter in every cell.
- The sensitivity of a stream or lake flux to a well rate is the **capture
  fraction** directly.
- The prediction well captures 1.032 of its pumping from the stream, and the lake
  gives up 0.013 of it less than before the well.
- Mapped over the grid, the same `wel6_q` sensitivity gives the capture
  fraction for a well placed anywhere, from the one backward solve. A lake with no
  outlet gives up less water to a well near it, not more.
- The per-period sensitivities show the aquifer's memory: recent pumping
  dominates the capture at any given time.
- Mapping the sensitivity to hydraulic conductivity shows which parts of the
  aquifer control the stream exchange, and where better data would be worth
  collecting.
- mf6adj warns when part of a package's derivative is not formed. Read those
  warnings, and check a measure with a small-perturbation two-run difference
  and a full water budget when they name something the model uses. Here the
  evapotranspiration UZF takes above a lowered water table is what the stream
  measure leaves out.